# Pandas from First Principles
## Notebook 3: Data Cleaning

---

**What you already know:** Series, DataFrame, `.loc`/`.iloc`, filtering, add/drop/rename columns.

**What you will learn here:** How to clean messy, real-world data — the skill you will use in literally every data project.

---

### Table of Contents

1. Why Data Cleaning Matters
2. Checking for Missing Values
3. Dropping Missing Values — `dropna()`
4. Filling Missing Values — `fillna()`
5. Checking and Removing Duplicates
6. Fixing Data Types
7. Replacing Values — `replace()`
8. Practice Questions

---

# 1. Why Data Cleaning Matters

## What is the problem?

Real-world data is almost never clean. When you receive a CSV from a client, download data from an API, or export from a database, you will commonly see:

- Missing values (blank cells, `NaN`, `None`, `"N/A"` strings)
- Duplicate rows (same record entered twice)
- Wrong data types (salary stored as string, date stored as text)
- Inconsistent values (`"delhi"`, `"Delhi"`, `"DELHI"` all meaning the same thing)
- Impossible values (age = -5, salary = 0)

If you run analysis on dirty data, you get **wrong answers** — and often no error message to warn you.

## Real-world analogy

Think of raw data like a form filled in by hand by 10,000 people. Some left fields blank. Some wrote "NA". Some wrote their age as text ("twenty five"). Some submitted the form twice. Before you can count or analyze anything, you have to fix the form first.

**Data cleaning is not glamorous. But it is 70–80% of a data analyst's actual work.**

---

In this notebook we will use this messy employee dataset throughout.

In [12]:
import pandas as pd
import numpy as np

# This dataset has intentional problems — we will fix them one by one
raw_data = {
    "name":       ["Alice", "Bob", "Carol", "Dave", "Eve", "Bob", "Frank", "Grace", "Hank", "Ivy"],
    "age":        [28, 34, None, 41, 30, 34, -5, 27, None, 32],
    "department": ["Engineering", "Marketing", "Engineering", None, "marketing", "Marketing", "HR", "Engineering", "HR", None],
    "salary":     ["85000", "72000", "91000", "68000", None, "72000", "0", "88000", "64000", "77000"],
    "city":       ["Delhi", "Mumbai", "Pune", "Delhi", "Bangalore", "Mumbai", "Delhi", None, "Pune", "Bangalore"],
    "join_date":  ["2020-03-15", "2019-07-22", "2021-01-10", "2018-11-05", "2022-06-30",
                   "2019-07-22", "2023-02-14", "2021-09-01", "2020-08-19", "2022-04-11"],
}

length = len(raw_data)
df = pd.DataFrame(data=raw_data,
                  index = range(1,len(raw_data['name'])+1))
display(df)
print("\nShape:", df.shape)

,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,None,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,None,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,None,77000,Bangalore,2022-04-11



Shape: (10, 6)


Can you spot the problems just by looking?

- `age` has `None` values and `-5` (impossible age)
- `department` has `None` values and inconsistent casing (`"marketing"` vs `"Marketing"`)
- `salary` is stored as **strings**, not numbers
- `salary` has `"0"` (invalid) and `None`
- `city` has a missing value
- Row 5 (Bob, index 5) looks like a **duplicate** of row 1 (Bob, index 1)

We will fix every one of these — step by step.

---

# 2. Checking for Missing Values

## What is it?

Before fixing missing values, you need to know **where** they are and **how many** there are.

You already know `isna()` from Notebook 1 (for Series). It works on DataFrames too — and gives you a table of True/False for every cell.

In [13]:
# .isna() on a DataFrame — returns True wherever a value is missing
print(df.isna())

     name    age  department  salary   city  join_date
1   False  False       False   False  False      False
2   False  False       False   False  False      False
3   False   True       False   False  False      False
4   False  False        True   False  False      False
5   False  False       False    True  False      False
6   False  False       False   False  False      False
7   False  False       False   False  False      False
8   False  False       False   False   True      False
9   False   True       False   False  False      False
10  False  False        True   False  False      False


That shows every cell — but it is hard to read. In practice, you want a **summary**.

In [17]:
# Count of missing values per column
print("Missing values per column:")
display(df.isna().sum().to_frame())

Missing values per column:


,0
name,0
age,2
department,2
salary,1
city,1
join_date,0


In [20]:
# Percentage of missing values per column
missing_pct = (df.isna().sum() / len(df) * 100).round(1)
print("Missing percentage per column:")
display(missing_pct.to_frame())

Missing percentage per column:


,0
name,0.0
age,20.0
department,20.0
salary,10.0
city,10.0
join_date,0.0


In [22]:
# Total missing values in the entire DataFrame
print("Total missing cells:", df.isna().sum().sum())

# Which rows have at least one missing value?
rows_with_missing = df[df.isna().any(axis=1)]
print("\nRows with at least one missing value:")
print(rows_with_missing)

Total missing cells: 6

Rows with at least one missing value:
     name   age   department salary       city   join_date
3   Carol   NaN  Engineering  91000       Pune  2021-01-10
4    Dave  41.0         None  68000      Delhi  2018-11-05
5     Eve  30.0    marketing   None  Bangalore  2022-06-30
8   Grace  27.0  Engineering  88000       None  2021-09-01
9    Hank   NaN           HR  64000       Pune  2020-08-19
10    Ivy  32.0         None  77000  Bangalore  2022-04-11


**Understanding `axis=1`:**

- `df.isna().any(axis=0)` → for each **column**, is there any missing value?
- `df.isna().any(axis=1)` → for each **row**, is there any missing value?

Think of it this way: `axis=1` means "look across the columns" (left to right), which gives you a result per row.

**Best practice:** Always run this missing value check right after loading any new dataset. Make it a habit before doing anything else.

---

# 3. Dropping Missing Values — `dropna()`

## What is it?

`dropna()` removes rows (or columns) that contain missing values.

## Syntax

```python
df.dropna(axis=0, how='any', thresh=None, subset=None)
```

## Parameters

| Parameter | What it does | Default |
|---|---|---|
| `axis` | `0` = drop rows, `1` = drop columns | `0` |
| `how` | `'any'` = drop if ANY value is missing. `'all'` = drop only if ALL values are missing | `'any'` |
| `thresh` | Keep rows with at least `thresh` non-missing values | `None` |
| `subset` | Check only these columns for missing values | `None` (check all) |

## Return value

Returns a **new** DataFrame. Original is unchanged.

In [24]:
# Default: drop any row that has at least one missing value
df_dropped = df.dropna()
print(f"Original rows: {len(df)}")
print(f"After dropna(): {len(df_dropped)}")
print(df_dropped)

Original rows: 10
After dropna(): 4
    name   age   department salary    city   join_date
1  Alice  28.0  Engineering  85000   Delhi  2020-03-15
2    Bob  34.0    Marketing  72000  Mumbai  2019-07-22
6    Bob  34.0    Marketing  72000  Mumbai  2019-07-22
7  Frank  -5.0           HR      0   Delhi  2023-02-14


We lost 6 rows! That is too aggressive. In real projects, dropping all rows with any missing value often loses too much data. You need to be smarter about it.

## 3.1 Drop only if specific columns are missing — `subset`

In [27]:
# Only drop a row if 'name' or 'salary' is missing
# A missing 'city' or 'department' is acceptable
df_smart_drop = df.dropna(subset=["name", "salary"])

print(f"After subset dropna: {len(df_smart_drop)} rows")
display(df_smart_drop)

After subset dropna: 9 rows


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,None,68000,Delhi,2018-11-05
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,None,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,None,77000,Bangalore,2022-04-11


## 3.2 Drop only if ALL values in a row are missing — `how='all'`

In [30]:
# Create a DataFrame with a completely empty row
df_with_empty_row = df.copy()
df_with_empty_row.loc[11] = [None, None, None, None, None, None]  # add empty row

print("With empty row added:")
print(df_with_empty_row.tail(3))

# how='all' only drops rows where EVERY column is NaN
df_cleaned = df_with_empty_row.dropna(how="all")
print(f"\nAfter dropna(how='all'): {len(df_cleaned)} rows (empty row removed)")

With empty row added:
    name   age department salary       city   join_date
9   Hank   NaN         HR  64000       Pune  2020-08-19
10   Ivy  32.0       None  77000  Bangalore  2022-04-11
11  None   NaN       None   None       None        None

After dropna(how='all'): 10 rows (empty row removed)


/var/folders/pk/b1n6lmf944b3fm6882d66scw0000gn/T/ipykernel_3400/2542850661.py:3: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_with_empty_row.loc[11] = [None, None, None, None, None, None]  # add empty row


## 3.3 Drop columns with too many missing values — `axis=1`

In [31]:
# Drop any column that has at least one missing value
df_col_drop = df.dropna(axis=1)
print("Columns remaining:", df_col_drop.columns.tolist())

Columns remaining: ['name', 'join_date']


In [ ]:
# More practical: drop columns where more than 30% of values are missing
threshold = len(df) * 0.70  # at least 70% of values must be present
df_thresh = df.dropna(axis=1, thresh=int(threshold))
print("Columns after thresh filter:", df_thresh.columns.tolist())

**When to use `dropna()` vs `fillna()`:**

| Situation | Use |
|---|---|
| Very few rows have missing data | `dropna()` — safe to lose them |
| Missing in critical columns (ID, key) | `dropna(subset=[...])` |
| Missing in less critical columns | `fillna()` — keep the row, fill the gap |
| Entire rows are empty | `dropna(how='all')` |
| Column has > 50% missing | `dropna(axis=1)` — column is not usable |

---

# 4. Filling Missing Values — `fillna()`

## What is it?

Instead of dropping rows, you replace missing values with something meaningful.

## Syntax

```python
df.fillna(value)
df["column"].fillna(value)
```

## Return value

Returns a **new** DataFrame or Series. Original is unchanged.

The key insight: **different columns often need different fill strategies.**
You should handle each column separately — do not fill everything with the same value.

In [32]:
# Start with a fresh copy
df_clean = df.copy()
print("Missing values before filling:")
print(df_clean.isna().sum())

Missing values before filling:
name          0
age           2
department    2
salary        1
city          1
join_date     0
dtype: int64


## 4.1 Fill each column with a specific value

In [35]:
# Fill 'city' missing values with 'Unknown'
display(df_clean)
df_clean["city"] = df_clean["city"].fillna("Unknown")

print("city column after fill:")
display(df_clean)

,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,None,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,Unknown,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,None,77000,Bangalore,2022-04-11


city column after fill:


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,None,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,Unknown,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,None,77000,Bangalore,2022-04-11


In [38]:
# Fill 'department' with the most common value (mode)
most_common_dept = df_clean["department"].mode()[0]  # mode() returns a Series, [0] = first value
print("Most common department:", most_common_dept)

df_clean["department"] = df_clean["department"].fillna(most_common_dept)
print("\ndepartment after fill:")
display(df_clean)

Most common department: Engineering

department after fill:


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,Engineering,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,Unknown,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,Engineering,77000,Bangalore,2022-04-11


In [40]:
# Fill 'age' with the median (better than mean for data that might be skewed)
# But first: we need to fix salary dtype so we can compute median properly
# For now, just fill age NaN with median of the valid ages

age_median = df_clean["age"].median()
print("Median age (of valid values):", age_median)

df_clean["age"] = df_clean["age"].fillna(age_median)
print("\nage after fill:")
display(df_clean)

Median age (of valid values): 31.0

age after fill:


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,31.0,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,Engineering,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,Unknown,2021-09-01
9,Hank,31.0,HR,64000,Pune,2020-08-19
10,Ivy,32.0,Engineering,77000,Bangalore,2022-04-11


## 4.2 Fill multiple columns at once using a dictionary

Instead of handling columns one by one, you can pass a dictionary to `fillna()` where each key is a column name.

In [42]:
# Fill multiple columns in one call
df2 = df.copy()

fill_values = {
    "city":       "Unknown",
    "department": "Not Assigned",
    "salary":     "0",
}

df2 = df2.fillna(fill_values)

print("After filling with dict:")
display(df2)

After filling with dict:


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,Not Assigned,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,0,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,Unknown,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,Not Assigned,77000,Bangalore,2022-04-11


## 4.3 Forward fill and backward fill

`ffill()` carries the last known value forward. `bfill()` carries the next known value backward.

These work well when rows have a natural order — like time series or sorted data.

In [43]:
# Example: daily temperature readings with some gaps
temp_data = pd.DataFrame({
    "day":  ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"],
    "temp": [22.5, None, None, 25.1, 23.8, None, 24.0],
})

print("Original:")
print(temp_data)

print("\nAfter ffill (carry last valid value forward):")
print(temp_data["temp"].ffill())

print("\nAfter bfill (carry next valid value backward):")
print(temp_data["temp"].bfill())

Original:
   day  temp
0  Mon  22.5
1  Tue   NaN
2  Wed   NaN
3  Thu  25.1
4  Fri  23.8
5  Sat   NaN
6  Sun  24.0

After ffill (carry last valid value forward):
0    22.5
1    22.5
2    22.5
3    25.1
4    23.8
5    23.8
6    24.0
Name: temp, dtype: float64

After bfill (carry next valid value backward):
0    22.5
1    25.1
2    25.1
3    25.1
4    23.8
5    24.0
6    24.0
Name: temp, dtype: float64


---

# 5. Checking and Removing Duplicates

## What is it?

Duplicate rows are rows where all (or most) column values are the same. They happen when:
- A form was submitted twice
- Data was merged from two sources and the same record appeared in both
- An API returned overlapping date ranges

Keeping duplicates inflates counts, sums, and averages — silently giving you wrong results.

## 5.1 `duplicated()` — find duplicate rows

### Syntax
```python
df.duplicated(subset=None, keep='first')
```

| Parameter | What it does | Default |
|---|---|---|
| `subset` | Check only these columns for duplicates | `None` (check all columns) |
| `keep` | `'first'` = mark all duplicates except first. `'last'` = except last. `False` = mark all | `'first'` |

Returns a **boolean Series** — `True` where a row is a duplicate.

In [44]:
# Find duplicate rows (checking all columns)
print("Duplicate mask:")
print(df.duplicated())

print("\nNumber of duplicate rows:", df.duplicated().sum())

Duplicate mask:
1     False
2     False
3     False
4     False
5     False
6      True
7     False
8     False
9     False
10    False
dtype: bool

Number of duplicate rows: 1


In [46]:
# Show the actual duplicate rows
print("Rows that are duplicates:")
display(df[df.duplicated()])

Rows that are duplicates:


,name,age,department,salary,city,join_date
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22


In [ ]:
# Check duplicates based on specific columns only
# Here: is the same name+department combination repeated?
print("Duplicate based on name + department:")
print(df[df.duplicated(subset=["name", "department"])])

## 5.2 `drop_duplicates()` — remove duplicate rows

### Syntax
```python
df.drop_duplicates(subset=None, keep='first')
```

Same parameters as `duplicated()`. Returns a **new** DataFrame.

In [49]:
# Remove duplicate rows — keeps the first occurrence, drops the rest
df_no_dups = df.drop_duplicates()

print(f"Before: {len(df)} rows")
print(f"After:  {len(df_no_dups)} rows")
display(df_no_dups)

Before: 10 rows
After:  9 rows


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
2,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,None,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,None,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,None,77000,Bangalore,2022-04-11


In [51]:
# Keep the LAST occurrence instead of the first
df_keep_last = df.drop_duplicates(keep="last")
print(f"Keeping last: {len(df_keep_last)} rows")
display(df_keep_last)

Keeping last: 9 rows


,name,age,department,salary,city,join_date
1,Alice,28.0,Engineering,85000,Delhi,2020-03-15
3,Carol,NaN,Engineering,91000,Pune,2021-01-10
4,Dave,41.0,None,68000,Delhi,2018-11-05
5,Eve,30.0,marketing,None,Bangalore,2022-06-30
6,Bob,34.0,Marketing,72000,Mumbai,2019-07-22
7,Frank,-5.0,HR,0,Delhi,2023-02-14
8,Grace,27.0,Engineering,88000,None,2021-09-01
9,Hank,NaN,HR,64000,Pune,2020-08-19
10,Ivy,32.0,None,77000,Bangalore,2022-04-11


In [52]:
# Drop duplicates based on specific columns only
# Useful when you want unique employees by name regardless of other columns
df_unique_names = df.drop_duplicates(subset=["name"], keep="first")
print(f"Unique by name: {len(df_unique_names)} rows")
display(df_unique_names[["name", "department", "salary"]])

Unique by name: 9 rows


,name,department,salary
1,Alice,Engineering,85000
2,Bob,Marketing,72000
3,Carol,Engineering,91000
4,Dave,None,68000
5,Eve,marketing,None
7,Frank,HR,0
8,Grace,Engineering,88000
9,Hank,HR,64000
10,Ivy,None,77000


**Interview note:** The order matters with `keep`. If you have two duplicate rows and one has a typo, use `keep='last'` if the last entry is the corrected one, or `keep='first'` if the first entry is correct. Always check your data before deciding.

---

# 6. Fixing Data Types

## What is the problem?

When you load a CSV, Pandas reads everything as text first and then tries to guess the type. It often gets it wrong:

- A column with mostly numbers but one `"N/A"` string → stored as `object`
- A date column → stored as `object` instead of `datetime`
- An integer column → stored as `float64` because it has missing values

Wrong types mean: you cannot do math on numbers stored as strings, you cannot filter by date properly, you waste memory.

## 6.1 Checking types — always do this first

In [53]:
# Look at what we have
print(df.dtypes)
print("\nNotice: 'salary' is object (string), 'age' is float64 (because of NaN), 'join_date' is object")

name           object
age           float64
department     object
salary         object
city           object
join_date      object
dtype: object

Notice: 'salary' is object (string), 'age' is float64 (because of NaN), 'join_date' is object


## 6.2 Converting to numeric — `pd.to_numeric()`

### Syntax
```python
pd.to_numeric(series, errors='raise')
```

| `errors` value | Behavior |
|---|---|
| `'raise'` (default) | Raise error on bad values |
| `'coerce'` | Turn bad values into `NaN` |
| `'ignore'` | Return original if conversion fails |

**Use `errors='coerce'` for real-world data.** It is the safe option.

In [54]:
# salary column is stored as string — convert to numeric
df_clean2 = df.copy()

print("Before conversion:")
print("dtype:", df_clean2["salary"].dtype)
print(df_clean2["salary"].tolist())

# Convert to numeric — None becomes NaN, valid strings become float
df_clean2["salary"] = pd.to_numeric(df_clean2["salary"], errors="coerce")

print("\nAfter conversion:")
print("dtype:", df_clean2["salary"].dtype)
print(df_clean2["salary"].tolist())

Before conversion:
dtype: object
['85000', '72000', '91000', '68000', None, '72000', '0', '88000', '64000', '77000']

After conversion:
dtype: float64
[85000.0, 72000.0, 91000.0, 68000.0, nan, 72000.0, 0.0, 88000.0, 64000.0, 77000.0]


In [58]:
# Now we can do math on it
print("Average salary:", df_clean2["salary"].mean().round())
print("Max salary:    ", df_clean2["salary"].max())

Average salary: 68556.0
Max salary:     91000.0


## 6.3 Converting to datetime — `pd.to_datetime()`

### Syntax
```python
pd.to_datetime(series, format=None, errors='raise')
```

Date columns are almost always loaded as strings. Converting them to `datetime` lets you extract the year, month, day, and do date arithmetic.

In [65]:
# join_date is stored as a string — convert to datetime
print("Before conversion:")
print("dtype:", df_clean2["join_date"].dtype)
print(df_clean2["join_date"].tolist())

df_clean2["join_date"] = pd.to_datetime(df_clean2["join_date"]).dt.date

print("\nAfter conversion:")
print("dtype:", df_clean2["join_date"].dtype)
print(df_clean2["join_date"].tolist())

Before conversion:
dtype: datetime64[ns]
[Timestamp('2020-03-15 00:00:00'), Timestamp('2019-07-22 00:00:00'), Timestamp('2021-01-10 00:00:00'), Timestamp('2018-11-05 00:00:00'), Timestamp('2022-06-30 00:00:00'), Timestamp('2019-07-22 00:00:00'), Timestamp('2023-02-14 00:00:00'), Timestamp('2021-09-01 00:00:00'), Timestamp('2020-08-19 00:00:00'), Timestamp('2022-04-11 00:00:00')]

After conversion:
dtype: object
[datetime.date(2020, 3, 15), datetime.date(2019, 7, 22), datetime.date(2021, 1, 10), datetime.date(2018, 11, 5), datetime.date(2022, 6, 30), datetime.date(2019, 7, 22), datetime.date(2023, 2, 14), datetime.date(2021, 9, 1), datetime.date(2020, 8, 19), datetime.date(2022, 4, 11)]


In [64]:
# Now you can extract parts of the date
df_clean2["join_year"]  = df_clean2["join_date"].dt.year
df_clean2["join_month"] = df_clean2["join_date"].dt.month

print(df_clean2[["name", "join_date", "join_year", "join_month"]])

     name  join_date  join_year  join_month
1   Alice 2020-03-15       2020           3
2     Bob 2019-07-22       2019           7
3   Carol 2021-01-10       2021           1
4    Dave 2018-11-05       2018          11
5     Eve 2022-06-30       2022           6
6     Bob 2019-07-22       2019           7
7   Frank 2023-02-14       2023           2
8   Grace 2021-09-01       2021           9
9    Hank 2020-08-19       2020           8
10    Ivy 2022-04-11       2022           4


**Note:** We will go deeper into date operations in Notebook 4 (String & DateTime). For now, just know that `pd.to_datetime()` is the right tool to convert date strings.

## 6.4 Converting with `.astype()`

Use `.astype()` when you know the data is already clean and just needs a type change.

In [66]:
# age is float64 because of NaN — after filling NaN, convert back to int
df_clean2["age"] = df_clean2["age"].fillna(df_clean2["age"].median())
df_clean2["age"] = df_clean2["age"].astype(int)

print("age dtype after conversion:", df_clean2["age"].dtype)
print(df_clean2["age"].tolist())

age dtype after conversion: int64
[28, 34, 31, 41, 30, 34, -5, 27, 31, 32]


**Common mistake:** Using `.astype(int)` before handling `NaN`. You cannot convert `NaN` to `int` — it will raise an error. Always fill or drop NaN first.

```python
# WRONG — will crash if any NaN exists
df["age"] = df["age"].astype(int)

# CORRECT — fill NaN first, then convert
df["age"] = df["age"].fillna(df["age"].median()).astype(int)
```

---

# 7. Replacing Values — `replace()`

## What is it?

`replace()` lets you swap specific values in a column (or entire DataFrame) with new values.

It is different from `fillna()` — `fillna()` only handles `NaN`. `replace()` handles any existing value.

## Syntax

```python
df["column"].replace(to_replace, value)
df.replace({"column": {old_value: new_value}})
```

## Return value

Returns a **new** Series or DataFrame. Original is unchanged.

In [67]:
# Problem: 'department' has inconsistent casing — 'marketing' vs 'Marketing'
print("Before fixing casing:")
print(df["department"].value_counts())

Before fixing casing:
department
Engineering    3
Marketing      2
HR             2
marketing      1
Name: count, dtype: int64


In [68]:
# Fix inconsistent casing using .str.title() — applies to the whole column at once
df_clean3 = df.copy()
df_clean3["department"] = df_clean3["department"].str.title()

print("After fixing casing:")
print(df_clean3["department"].value_counts())

After fixing casing:
department
Engineering    3
Marketing      3
Hr             2
Name: count, dtype: int64


In [ ]:
# Problem: salary has '0' which is an invalid salary
# Replace 0 with NaN so we can handle it properly
df_clean3["salary"] = pd.to_numeric(df_clean3["salary"], errors="coerce")
df_clean3["salary"] = df_clean3["salary"].replace(0, np.nan)

print("Salary after replacing 0 with NaN:")
print(df_clean3["salary"])

In [ ]:
# Replace multiple values at once using a dictionary
city_corrections = {
    "Bombay": "Mumbai",
    "Bengaluru": "Bangalore",
    "New Delhi": "Delhi",
}

# This fixes city name variations — replace() handles all of them in one call
sample_cities = pd.Series(["Delhi", "Bombay", "Bengaluru", "Pune", "New Delhi", "Mumbai"])
cleaned_cities = sample_cities.replace(city_corrections)

print("Before:", sample_cities.tolist())
print("After: ", cleaned_cities.tolist())

In [ ]:
# Replace across multiple columns in one call
df_example = pd.DataFrame({
    "answer_q1": ["Yes", "No", "Yes", "N/A", "No"],
    "answer_q2": ["No", "Yes", "N/A", "Yes", "No"],
})

# Convert all 'Yes' to 1, 'No' to 0, 'N/A' to NaN — across all columns
df_encoded = df_example.replace({"Yes": 1, "No": 0, "N/A": np.nan})
print(df_encoded)

## 7.1 Fixing impossible values

Some values are not `NaN` but are still invalid — like `age = -5` or `salary = 0`. You need to handle these too.

In [ ]:
# Fix impossible age values: age < 0 or age > 100 are impossible
df_clean3["age"] = pd.to_numeric(df_clean3["age"], errors="coerce")

# Replace invalid ages with NaN using .where()
# .where(condition) keeps values where condition is True, replaces rest with NaN
df_clean3["age"] = df_clean3["age"].where(
    (df_clean3["age"] > 0) & (df_clean3["age"] < 100)
)

print("Ages after fixing impossible values:")
print(df_clean3[["name", "age"]])

**How `.where()` works:**

```python
series.where(condition)
```

- Where `condition` is `True` → keep the original value
- Where `condition` is `False` → replace with `NaN`

It is the opposite of boolean filtering. Filtering keeps only True rows. `.where()` keeps the True values in place and turns False values into NaN.

## 7.2 Putting it all together — a complete cleaning pipeline

In [69]:
# Complete pipeline: clean the raw_data from top to bottom

# Start fresh
clean = pd.DataFrame(raw_data).copy()

# Step 1: Remove exact duplicate rows
clean = clean.drop_duplicates()

# Step 2: Fix salary — convert to numeric, replace 0 with NaN
clean["salary"] = pd.to_numeric(clean["salary"], errors="coerce")
clean["salary"] = clean["salary"].replace(0, np.nan)
clean["salary"] = clean["salary"].fillna(clean["salary"].median())

# Step 3: Fix age — convert, remove impossible values, fill with median
clean["age"] = pd.to_numeric(clean["age"], errors="coerce")
clean["age"] = clean["age"].where((clean["age"] > 0) & (clean["age"] < 100))
clean["age"] = clean["age"].fillna(clean["age"].median()).astype(int)

# Step 4: Fix department — standardize casing, fill missing with mode
clean["department"] = clean["department"].str.title()
clean["department"] = clean["department"].fillna(clean["department"].mode()[0])

# Step 5: Fix city — fill missing with 'Unknown'
clean["city"] = clean["city"].fillna("Unknown")

# Step 6: Convert join_date to datetime
clean["join_date"] = pd.to_datetime(clean["join_date"])

print("=== Cleaned DataFrame ===")
print(clean)
print("\nMissing values remaining:")
print(clean.isna().sum())
print("\nDtypes:")
print(clean.dtypes)

=== Cleaned DataFrame ===
    name  age   department   salary       city  join_date
0  Alice   28  Engineering  85000.0      Delhi 2020-03-15
1    Bob   34    Marketing  72000.0     Mumbai 2019-07-22
2  Carol   31  Engineering  91000.0       Pune 2021-01-10
3   Dave   41  Engineering  68000.0      Delhi 2018-11-05
4    Eve   30    Marketing  77000.0  Bangalore 2022-06-30
6  Frank   31           Hr  77000.0      Delhi 2023-02-14
7  Grace   27  Engineering  88000.0    Unknown 2021-09-01
8   Hank   31           Hr  64000.0       Pune 2020-08-19
9    Ivy   32  Engineering  77000.0  Bangalore 2022-04-11

Missing values remaining:
name          0
age           0
department    0
salary        0
city          0
join_date     0
dtype: int64

Dtypes:
name                  object
age                    int64
department            object
salary               float64
city                  object
join_date     datetime64[ns]
dtype: object


Zero missing values. Correct dtypes. Clean data — ready for analysis.

**This pipeline approach — step by step, one problem at a time — is exactly how real data cleaning works in production.**

---

# 8. Practice Questions

---

## Level 1 — Easy

### Q1 — Finding missing values

Using the DataFrame below:
- a) How many missing values are there in each column?
- b) What percentage of `score` values are missing?
- c) Which rows have at least one missing value?
- d) Are there any columns where ALL values are missing?

In [99]:
import pandas as pd
import numpy as np

exam_df = pd.DataFrame(data = {
    "student":  ["Riya", "Arjun", "Priya", "Karan", "Meera", "Dev"],
    "score":    [88, None, 74, None, 91, 63],
    "subject":  ["Math", "Science", None, "Math", "Science", "Math"],
    "grade":    ["A", None, "B", None, "A", "C"],},
    index= [1,2,3,4,5,6])

display(exam_df)

# Q1 — Your solution here
print('Missing Count:')
display(exam_df.isna().sum().to_frame(name='Missing Count'))
print('Score Missing:')
print(f'% missing in score: {exam_df['score'].isna().mean()*100:.1f}%')
print("\nc) Rows with at least one missing:")
display(exam_df[exam_df.isna().any(axis=1)])
print("\nd) Columns where ALL values missing:", exam_df.columns[exam_df.isna().all()].tolist())


,student,score,subject,grade
1,Riya,88.0,Math,A
2,Arjun,NaN,Science,None
3,Priya,74.0,None,B
4,Karan,NaN,Math,None
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


Missing Count:


,Missing Count
student,0
score,2
subject,1
grade,2


Score Missing:
% missing in score: 33.3%

c) Rows with at least one missing:


,student,score,subject,grade
2,Arjun,NaN,Science,None
3,Priya,74.0,None,B
4,Karan,NaN,Math,None



d) Columns where ALL values missing: []


### Q2 — dropna()

Using `exam_df` from Q1:
- a) Drop all rows that have any missing value
- b) Drop rows only where `score` is missing
- c) Drop rows where BOTH `score` AND `grade` are missing
- d) How many rows remain in each case?

In [106]:
display(exam_df)
print("Dropped missing rows:")
display(exam_df.dropna())
print('Dropped missing score:')
display(exam_df.dropna(subset=['score']))
print('Dropped Missing score and grade:')
display(exam_df.dropna(subset=['score','grade']))


,student,score,subject,grade
1,Riya,88.0,Math,A
2,Arjun,NaN,Science,None
3,Priya,74.0,None,B
4,Karan,NaN,Math,None
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


Dropped missing rows:


,student,score,subject,grade
1,Riya,88.0,Math,A
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


Dropped missing score:


,student,score,subject,grade
1,Riya,88.0,Math,A
3,Priya,74.0,None,B
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


Dropped Missing score and grade:


,student,score,subject,grade
1,Riya,88.0,Math,A
3,Priya,74.0,None,B
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


### Q3 — fillna()

Using `exam_df` from Q1:
- a) Fill missing `score` values with the **mean** score
- b) Fill missing `subject` with `'Unknown'`
- c) Fill missing `grade` with the **most common** grade
- d) Print the result — there should be no missing values

In [119]:
print('Full Dataset:')
display(exam_df)

mean_score = exam_df['score'].mean()
exam_df['score'] = exam_df['score'].fillna(mean_score)
print('Filled Score By mean:')
display(exam_df)

most_common_grade = exam_df['grade'].mode()[0]
exam_df['grade'] = exam_df['grade'].fillna(most_common_grade)
print('Filled most common grade:')
display(exam_df)

Full Dataset:


,student,score,subject,grade
1,Riya,88.0,Math,A
2,Arjun,79.0,Science,A
3,Priya,74.0,None,B
4,Karan,79.0,Math,A
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


Filled Score By mean:


,student,score,subject,grade
1,Riya,88.0,Math,A
2,Arjun,79.0,Science,A
3,Priya,74.0,None,B
4,Karan,79.0,Math,A
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


Filled most common grade:


,student,score,subject,grade
1,Riya,88.0,Math,A
2,Arjun,79.0,Science,A
3,Priya,74.0,None,B
4,Karan,79.0,Math,A
5,Meera,91.0,Science,A
6,Dev,63.0,Math,C


### Q4 — Duplicates

Using the DataFrame below:
- a) Find how many duplicate rows exist
- b) Show the duplicate rows
- c) Remove duplicates, keeping the first occurrence
- d) Remove duplicates based on `email` column only (a person registered twice with same email)

In [136]:
users_df = pd.DataFrame({
    "name":  ["Alice", "Bob", "Alice", "Carol", "Bob", "Dave", "Alice"],
    "email": ["alice@gmail.com", "bob@gmail.com", "alice@gmail.com",
               "carol@yahoo.com", "bob@gmail.com", "dave@gmail.com", "alice@gmail.com"],
    "age":   [25, 30, 25, 28, 30, 35, 26],},
    index=[1,2,3,4,5,6,7]
)
display(users_df)

print(f'no of duplicate rows : {len(users_df[users_df.duplicated(keep='first')])}\n')
print('Duplicate Rows:' )
display(users_df[users_df.duplicated(keep='first')])
print('Removed Duplicates:')
display(users_df.drop_duplicates(keep='first'))
print('Dropped duplicate by email:')
display(users_df.drop_duplicates(subset='email',keep='first'))
# Q4 — Your solution here


,name,email,age
1,Alice,alice@gmail.com,25
2,Bob,bob@gmail.com,30
3,Alice,alice@gmail.com,25
4,Carol,carol@yahoo.com,28
5,Bob,bob@gmail.com,30
6,Dave,dave@gmail.com,35
7,Alice,alice@gmail.com,26


no of duplicate rows : 2

Duplicate Rows:


,name,email,age
3,Alice,alice@gmail.com,25
5,Bob,bob@gmail.com,30


Removed Duplicates:


,name,email,age
1,Alice,alice@gmail.com,25
2,Bob,bob@gmail.com,30
4,Carol,carol@yahoo.com,28
6,Dave,dave@gmail.com,35
7,Alice,alice@gmail.com,26


Dropped duplicate by email:


,name,email,age
1,Alice,alice@gmail.com,25
2,Bob,bob@gmail.com,30
4,Carol,carol@yahoo.com,28
6,Dave,dave@gmail.com,35


---

## Level 2 — Medium

### Q5 — Fixing data types

The DataFrame below has type problems. Fix them all:
- a) Convert `revenue` to numeric (some values are strings, some invalid)
- b) Convert `order_date` to datetime
- c) Extract the **month** from `order_date` into a new column called `order_month`
- d) Convert `units` to integer (fill any NaN with 0 first)

In [176]:
orders_df = pd.DataFrame({
    "order_id":   [101, 102, 103, 104, 105],
    "revenue":    ["5000", "12000", "bad_value", "8500", None],
    "units":      [10, None, 25, 17, 8],
    "order_date": ["2024-01-15", "2024-02-10", "2024-02-28", "2024-03-05", "2024-03-22"],
})

display(orders_df)
print("Original dtypes:")
display(orders_df.dtypes.to_frame(name='DataType'))
print()

orders_df['revenue'] = pd.to_numeric(orders_df['revenue'],errors='coerce')
display(orders_df)

orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
display(orders_df)
orders_df['order_month'] = orders_df["order_date"].dt.month
display(orders_df)

orders_df['units'] = orders_df['units'].fillna(0).astype(int)
display(orders_df)



# Q5 — Your solution here


,order_id,revenue,units,order_date
0,101,5000,10.0,2024-01-15
1,102,12000,NaN,2024-02-10
2,103,bad_value,25.0,2024-02-28
3,104,8500,17.0,2024-03-05
4,105,None,8.0,2024-03-22


Original dtypes:


,DataType
order_id,int64
revenue,object
units,float64
order_date,object


,order_id,revenue,units,order_date
0,101,5000.0,10.0,2024-01-15
1,102,12000.0,NaN,2024-02-10
2,103,NaN,25.0,2024-02-28
3,104,8500.0,17.0,2024-03-05
4,105,NaN,8.0,2024-03-22


,order_id,revenue,units,order_date
0,101,5000.0,10.0,2024-01-15
1,102,12000.0,NaN,2024-02-10
2,103,NaN,25.0,2024-02-28
3,104,8500.0,17.0,2024-03-05
4,105,NaN,8.0,2024-03-22


,order_id,revenue,units,order_date,order_month
0,101,5000.0,10.0,2024-01-15,1
1,102,12000.0,NaN,2024-02-10,2
2,103,NaN,25.0,2024-02-28,2
3,104,8500.0,17.0,2024-03-05,3
4,105,NaN,8.0,2024-03-22,3


,order_id,revenue,units,order_date,order_month
0,101,5000.0,10,2024-01-15,1
1,102,12000.0,0,2024-02-10,2
2,103,NaN,25,2024-02-28,2
3,104,8500.0,17,2024-03-05,3
4,105,NaN,8,2024-03-22,3


### Q6 — replace() and value standardization

The `feedback` column below has many variations of the same answers. Standardize them:
- a) Replace all variations of 'yes' (`'Yes'`, `'YES'`, `'y'`, `'Y'`) with `1`
- b) Replace all variations of 'no' (`'No'`, `'NO'`, `'n'`, `'N'`) with `0`
- c) Replace `'maybe'` and `'Maybe'` with `NaN`
- d) Count how many 1s, 0s, and NaNs are in the final column

In [177]:
feedback_df = pd.DataFrame({
    "respondent": ["P1", "P2", "P3", "P4", "P5", "P6", "P7", "P8", "P9", "P10"],
    "feedback":   ["Yes", "no", "YES", "maybe", "Y", "N", "Yes", "Maybe", "NO", "y"],
})

display(feedback_df)

# Q6 — Your solution here


,respondent,feedback
0,P1,Yes
1,P2,no
2,P3,YES
3,P4,maybe
4,P5,Y
5,P6,N
6,P7,Yes
7,P8,Maybe
8,P9,NO
9,P10,y


### Q7 — Full cleaning pipeline

Clean this raw product data from top to bottom. The goal is zero missing values and correct dtypes.

- a) Remove exact duplicate rows
- b) Convert `price` to numeric — invalid values → NaN
- c) Remove rows where `price` is NaN or zero (invalid product)
- d) Fill missing `category` with `'Uncategorized'`
- e) Fix `in_stock` column: replace `'Yes'/'y'/'true'` with `True` and `'No'/'n'/'false'` with `False`
- f) Print final DataFrame with dtypes

In [161]:
products_raw = pd.DataFrame({
    "product":   ["Laptop", "Mouse", "Laptop", "Monitor", "Keyboard", "Webcam", "Mouse"],
    "price":     ["75000", "1200", "75000", "bad", "2500", "0", "1200"],
    "category":  ["Electronics", None, "Electronics", "Electronics", None, "Electronics", None],
    "in_stock":  ["Yes", "true", "Yes", "No", "y", "false", "true"],},
    index=[1,2,3,4,5,6,7])

print('Row Data:')
display(products_raw)

products = products_raw.drop_duplicates()
print('Dropped Duplicates:')
display(products)

print('Changed to Numeric:')
products['price'] = pd.to_numeric(products['price'],errors='coerce')
display(products)

products = products[(products["price"].notna()) & (products["price"] != 0)]
display(products)

print('Filling Missing Category: ')

products['categoory'] = products['category'].fillna('uncategorized')
display(products)

true_vals  = ["Yes", "y", "true"]
false_vals = ["No", "n", "false"]
products["in_stock"] = prod["in_stock"].replace(
    {v: True for v in true_vals} | {v: False for v in false_vals}
)

display(products)


# Q7 — Your solution here


Row Data:


,product,price,category,in_stock
1,Laptop,75000,Electronics,Yes
2,Mouse,1200,None,true
3,Laptop,75000,Electronics,Yes
4,Monitor,bad,Electronics,No
5,Keyboard,2500,None,y
6,Webcam,0,Electronics,false
7,Mouse,1200,None,true


Dropped Duplicates:


,product,price,category,in_stock
1,Laptop,75000,Electronics,Yes
2,Mouse,1200,None,true
4,Monitor,bad,Electronics,No
5,Keyboard,2500,None,y
6,Webcam,0,Electronics,false


Changed to Numeric:


...

,product,price,category,in_stock
1,Laptop,75000.0,Electronics,Yes
2,Mouse,1200.0,None,true
4,Monitor,NaN,Electronics,No
5,Keyboard,2500.0,None,y
6,Webcam,0.0,Electronics,false


,product,price,category,in_stock
1,Laptop,75000.0,Electronics,Yes
2,Mouse,1200.0,None,true
5,Keyboard,2500.0,None,y


Filling Missing Category: 


,product,price,category,in_stock,categoory
1,Laptop,75000.0,Electronics,Yes,Electronics
2,Mouse,1200.0,None,true,uncategorized
5,Keyboard,2500.0,None,y,uncategorized


,product,price,category,in_stock,categoory
1,Laptop,75000.0,Electronics,True,Electronics
2,Mouse,1200.0,None,NaN,uncategorized
5,Keyboard,2500.0,None,NaN,uncategorized


---

## Answer Key

In [88]:
# ── Q1 Answer ────────────────────────────────────────────────────────────────
exam_df = pd.DataFrame({
    "student":  ["Riya", "Arjun", "Priya", "Karan", "Meera", "Dev"],
    "score":    [88, None, 74, None, 91, 63],
    "subject":  ["Math", "Science", None, "Math", "Science", "Math"],
    "grade":    ["A", None, "B", None, "A", "C"],
})

print("a) Missing per column:")
print(exam_df.isna().sum())

print("\nb) % missing in score:", f"{exam_df['score'].isna().mean()*100:.1f}%")

print("\nc) Rows with at least one missing:")
print(exam_df[exam_df.isna().any(axis=1)])

print("\nd) Columns where ALL values missing:", exam_df.columns[exam_df.isna().all()].tolist())

a) Missing per column:
student    0
score      2
subject    1
grade      2
dtype: int64

b) % missing in score: 33.3%

c) Rows with at least one missing:
  student  score  subject grade
1   Arjun    NaN  Science  None
2   Priya   74.0     None     B
3   Karan    NaN     Math  None

d) Columns where ALL values missing: []


In [ ]:
# ── Q2 Answer ────────────────────────────────────────────────────────────────
a = exam_df.dropna()
b = exam_df.dropna(subset=["score"])
c = exam_df.dropna(subset=["score", "grade"], how="all")

print(f"a) Drop any missing:        {len(a)} rows remain")
print(f"b) Drop only score missing: {len(b)} rows remain")
print(f"c) Drop both missing:       {len(c)} rows remain")

In [ ]:
# ── Q3 Answer ────────────────────────────────────────────────────────────────
exam_filled = exam_df.copy()
exam_filled["score"]   = exam_filled["score"].fillna(exam_filled["score"].mean())
exam_filled["subject"] = exam_filled["subject"].fillna("Unknown")
exam_filled["grade"]   = exam_filled["grade"].fillna(exam_filled["grade"].mode()[0])

print("Filled DataFrame:")
print(exam_filled)
print("\nAny missing?", exam_filled.isna().any().any())

In [ ]:
# ── Q4 Answer ────────────────────────────────────────────────────────────────
users_df = pd.DataFrame({
    "name":  ["Alice","Bob","Alice","Carol","Bob","Dave","Alice"],
    "email": ["alice@gmail.com","bob@gmail.com","alice@gmail.com",
               "carol@yahoo.com","bob@gmail.com","dave@gmail.com","alice@gmail.com"],
    "age":   [25, 30, 25, 28, 30, 35, 26],
})

print("a) Duplicate count:", users_df.duplicated().sum())
print("\nb) Duplicate rows:")
print(users_df[users_df.duplicated()])
print("\nc) After drop_duplicates (keep first):")
print(users_df.drop_duplicates())
print("\nd) Unique by email:")
print(users_df.drop_duplicates(subset=["email"], keep="first"))

In [174]:
# ── Q5 Answer ────────────────────────────────────────────────────────────────
orders_df = pd.DataFrame({
    "order_id":   [101, 102, 103, 104, 105],
    "revenue":    ["5000", "12000", "bad_value", "8500", None],
    "units":      [10, None, 25, 17, 8],
    "order_date": ["2024-01-15","2024-02-10","2024-02-28","2024-03-05","2024-03-22"],
})

orders_df["revenue"]     = pd.to_numeric(orders_df["revenue"], errors="coerce")
orders_df["order_date"]  = pd.to_datetime(orders_df["order_date"])
orders_df["order_month"] = orders_df["order_date"].dt.month
orders_df["units"]       = orders_df["units"].fillna(0).astype(int)

print(orders_df)
print("\ndtypes:")
print(orders_df.dtypes)

   order_id  revenue  units order_date  order_month
0       101   5000.0     10 2024-01-15            1
1       102  12000.0      0 2024-02-10            2
2       103      NaN     25 2024-02-28            2
3       104   8500.0     17 2024-03-05            3
4       105      NaN      8 2024-03-22            3

dtypes:
order_id                int64
revenue               float64
units                   int64
order_date     datetime64[ns]
order_month             int32
dtype: object


In [ ]:
# ── Q6 Answer ────────────────────────────────────────────────────────────────
feedback_df = pd.DataFrame({
    "respondent": ["P1","P2","P3","P4","P5","P6","P7","P8","P9","P10"],
    "feedback":   ["Yes","no","YES","maybe","Y","N","Yes","Maybe","NO","y"],
})

yes_values   = ["Yes", "YES", "y", "Y"]
no_values    = ["No", "NO", "n", "N"]
maybe_values = ["maybe", "Maybe"]

feedback_df["feedback"] = feedback_df["feedback"].replace(
    {v: 1 for v in yes_values} |
    {v: 0 for v in no_values} |
    {v: np.nan for v in maybe_values}
)

print(feedback_df)
print("\nValue counts:")
print(feedback_df["feedback"].value_counts(dropna=False))

In [142]:
# ── Q7 Answer ────────────────────────────────────────────────────────────────
products_raw = pd.DataFrame({
    "product":   ["Laptop","Mouse","Laptop","Monitor","Keyboard","Webcam","Mouse"],
    "price":     ["75000","1200","75000","bad","2500","0","1200"],
    "category":  ["Electronics",None,"Electronics","Electronics",None,"Electronics",None],
    "in_stock":  ["Yes","true","Yes","No","y","false","true"],
})

prod = products_raw.copy()

# a) Remove exact duplicates
prod = prod.drop_duplicates()

# b) Convert price to numeric
prod["price"] = pd.to_numeric(prod["price"], errors="coerce")

# c) Remove rows where price is NaN or 0
prod = prod[prod["price"].notna() & (prod["price"] > 0)]

# d) Fill missing category
prod["category"] = prod["category"].fillna("Uncategorized")

# e) Standardize in_stock
true_vals  = ["Yes", "y", "true"]
false_vals = ["No", "n", "false"]
prod["in_stock"] = prod["in_stock"].replace(
    {v: True for v in true_vals} | {v: False for v in false_vals}
)

# f) Print result
print("Cleaned products:")
print(prod)
print("\ndtypes:")
print(prod.dtypes)
print("\nMissing values:")
print(prod.isna().sum())

Cleaned products:
    product    price       category  in_stock
0    Laptop  75000.0    Electronics      True
1     Mouse   1200.0  Uncategorized      True
4  Keyboard   2500.0  Uncategorized      True

dtypes:
product      object
price       float64
category     object
in_stock       bool
dtype: object

Missing values:
product     0
price       0
category    0
in_stock    0
dtype: int64


/var/folders/pk/b1n6lmf944b3fm6882d66scw0000gn/T/ipykernel_3400/3442782738.py:26: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  prod["in_stock"] = prod["in_stock"].replace(


---

## Quick Revision

| Topic | Key point |
|---|---|
| Check missing | `df.isna().sum()` — count per column. `df.isna().any(axis=1)` — rows with missing |
| `dropna()` | `how='any'` (default), `how='all'`, `subset=[cols]`, `axis=1` for columns |
| `fillna()` | Fill with scalar, dict (per column), `ffill()`, `bfill()` |
| `duplicated()` | Returns True/False mask. `subset=` to check specific columns |
| `drop_duplicates()` | Removes duplicates. `keep='first'` or `'last'`. Returns new DataFrame |
| `pd.to_numeric()` | Use `errors='coerce'` for real data. Bad values → NaN |
| `pd.to_datetime()` | Converts string dates to datetime type |
| `astype()` | Safe only when data is already clean. Fill NaN before converting to int |
| `replace()` | Swap specific values. Accepts dict for multiple replacements |
| `.where(cond)` | Keep True values, replace False with NaN |

## 3 Interview Questions

**Q1:** What is the difference between `dropna()` and `fillna()`? When would you choose one over the other?

**Q2:** You have a salary column stored as strings with some `"N/A"` values mixed in. How would you convert it to a usable numeric column?

**Q3:** What does `df.duplicated()` return? How is it different from `df.drop_duplicates()`?

---

## What's Next?

**Notebook 4 — String & DateTime:** Working with text data using `.str` methods, and extracting useful information from date columns using `.dt`.